# Column with node-to-surface constraints — v4

Same model, solver setup, and base-fixity trick (`zeroLength` to ground) as v3. **New in v4**: after the static analysis, the displacement field is pushed into a `Results` object and visualized through apeGmsh's `Results.viewer()`.

## The result pipeline

```
OpenSees  →  numpy arrays  →  Results.from_fem(fem, point_data={...})  →  results.viewer()
```

Fields attached in this notebook:

| name | shape | description |
|------|-------|-------------|
| `Displacement` | `(N, 3)` | nodal displacement vector — the viewer auto-detects this for the deformed-shape toggle |
| `\|U\|`        | `(N,)`  | displacement magnitude for contour colouring |
| `Ux`, `Uy`, `Uz` | `(N,)` | translation components for individual contour views |

Node ordering follows `fem.nodes.ids` — `Results.from_fem` remaps everything to 0-based VTK indices internally, so we just iterate `fem.nodes.ids` in order and call `ops.nodeDisp(int(tag))` for each.

Note: `fem.nodes.ids` contains only the 646 solid/reference nodes — the 44 constraint-generated phantom nodes live under `fem.nodes.constraints` and are **not** in the viewer mesh. That's fine: they exist only to implement the constraint chain, and OpenSees handles their kinematics invisibly.

In [1]:
from apeGmsh import apeGmsh, Part, Results
from apeGmsh import FEMData
from apeGmsh.sections import W_solid
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
m1 = apeGmsh(model_name='beam_column', verbose=False)
m1.begin()

column = m1.sections.W_solid(bf=150, tf=20, h=300, tw=10, length=2000, label='column')

m1.model.selection.select_volumes().to_physical(name='pg_column')

base = m1.model.geometry.add_point(x=0, y=0, z=0,    lc=100, label='base')
top  = m1.model.geometry.add_point(x=0, y=0, z=2000, lc=100, label='top')

m1.constraints.node_to_surface('base', column.labels.start_face)
m1.constraints.node_to_surface('top',  column.labels.end_face)

m1.loads.point(
    target='top',
    force_xyz=[0, 1000, 0],
    moment_xyz=[0, 0, 0],
)

m1.mesh.sizing.set_global_size(100)
m1.mesh.generation.generate(dim=3)

fem = m1.mesh.queries.get_fem_data(remove_orphans=True)

m1.model.viewer()
m1.mesh.viewer()

m1.end()

## OpenSees model + analysis (same as v3)

- `ndf = 6` global, solid column nodes overridden to `ndf = 3`.
- Grounded helper node at `GROUND_TAG` fixed in 6 DOFs, connected to the base reference point via a stiff 6-DOF `zeroLength` spring — so `nodeReaction` works directly on the ground node.
- `constraints('Penalty')` because the phantom nodes are daisy-chained (`rigidLink` slave + `equalDOF` master), which breaks `Transformation`.

In [3]:
import openseespy.opensees as ops

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

ops.timeSeries('Linear', 1)

# --- Nodes -----------------------------------------------------------------
for nid, xyz in fem.nodes.get(target='pg_column'):
    ops.node(nid, *xyz, '-ndf', 3)

ref = fem.nodes.get(target=['base', 'top'])
for nid, xyz in ref:
    ops.node(nid, *xyz)

for nid, xyz in fem.nodes.constraints.phantom_nodes():
    ops.node(nid, *xyz)

# --- Grounded helper node + zeroLength spring at the base ------------------
base_id = int(next(iter(fem.nodes.get_ids(target='base'))))
base_xyz = next(xyz for nid, xyz in ref if nid == base_id)

GROUND_TAG = 10_000
ops.node(GROUND_TAG, *base_xyz)
ops.fix(GROUND_TAG, 1, 1, 1, 1, 1, 1)

K_SPRING = 1e14
for i in range(6):
    ops.uniaxialMaterial('Elastic', 100 + i, K_SPRING)

ZL_TAG = 20_000
ops.element('zeroLength', ZL_TAG, GROUND_TAG, base_id,
            '-mat', 100, 101, 102, 103, 104, 105,
            '-dir', 1, 2, 3, 4, 5, 6)

# --- Material + tet elements -----------------------------------------------
E  = 200e3
nu = 0.3
ops.nDMaterial('ElasticIsotropic', 1, E, nu)

for group in fem.elements.get(element_type='tet4'):
    for eid, conn in group:
        ops.element('FourNodeTetrahedron', int(eid), *[int(n) for n in conn], 1)

# --- MP constraints --------------------------------------------------------
for master, slaves in fem.nodes.constraints.rigid_link_groups():
    for slave in slaves:
        ops.rigidLink('beam', int(master), int(slave))

for pair in fem.nodes.constraints.equal_dofs():
    ops.equalDOF(int(pair.master_node), int(pair.slave_node), *pair.dofs)

# --- Loads -----------------------------------------------------------------
ops.pattern('Plain', 1, 1)
for load in fem.nodes.loads.by_kind('nodal'):
    fx, fy, fz = load.force_xyz  or (0.0, 0.0, 0.0)
    mx, my, mz = load.moment_xyz or (0.0, 0.0, 0.0)
    ops.load(int(load.node_id), fx, fy, fz, mx, my, mz)

# --- Analysis --------------------------------------------------------------
ops.constraints('Penalty', 1e15, 1e15)
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-6, 10)
ops.algorithm('Linear')
ops.integrator('LoadControl', 0.1)
ops.analysis('Static')

for i in range(10):
    ok = ops.analyze(1)
    print(f'Step {i+1}: {"ok" if ok == 0 else f"failed ({ok})"}')

Step 1: ok
Step 2: ok
Step 3: ok
Step 4: ok
Step 5: ok
Step 6: ok
Step 7: ok


Step 8: ok
Step 9: ok
Step 10: ok


## Base reaction (via the grounded zeroLength)

Expected from equilibrium: `Fy = -1000 N`, `Mx = +2·10⁶ N·mm`.

In [4]:
ops.reactions()

rxn = ops.nodeReaction(GROUND_TAG)

print(f'nodeReaction(ground = {GROUND_TAG}):')
print(f'  Fx = {rxn[0]:+.4e}   Fy = {rxn[1]:+.4e}   Fz = {rxn[2]:+.4e}')
print(f'  Mx = {rxn[3]:+.4e}   My = {rxn[4]:+.4e}   Mz = {rxn[5]:+.4e}')
print(f'\nExpected:  Fy = -1.0000e+03   Mx = +2.0000e+06')

nodeReaction(ground = 10000):
  Fx = +6.3563e-05   Fy = -1.0000e+03   Fz = -6.9247e-05
  Mx = +2.0000e+06   My = +6.3709e-02   Mz = +3.7051e+00

Expected:  Fy = -1.0000e+03   Mx = +2.0000e+06


## Extract the displacement field

Build an `(N, 3)` array of displacements following the `fem.nodes.ids` order. For nodes that were created with `-ndf 3` (the solid column nodes) `ops.nodeDisp` returns 3 values; for the 6-DOF reference points we only keep the translation components. `Results.from_fem` consumes the array as-is.

In [5]:
n_nodes = len(fem.nodes.ids)
disp = np.zeros((n_nodes, 3), dtype=np.float64)

for i, nid in enumerate(fem.nodes.ids):
    d = ops.nodeDisp(int(nid))
    disp[i, 0] = d[0]
    disp[i, 1] = d[1]
    disp[i, 2] = d[2]

u_mag = np.linalg.norm(disp, axis=1)

print(f'Extracted displacements for {n_nodes} nodes.')
print(f'  ux range: [{disp[:,0].min():+.4e}, {disp[:,0].max():+.4e}]')
print(f'  uy range: [{disp[:,1].min():+.4e}, {disp[:,1].max():+.4e}]')
print(f'  uz range: [{disp[:,2].min():+.4e}, {disp[:,2].max():+.4e}]')
print(f'  |U| max:  {u_mag.max():+.4e}')

Extracted displacements for 646 nodes.
  ux range: [-2.6994e-04, +2.3853e-04]
  uy range: [+5.7077e-12, +7.8756e-02]
  uz range: [-9.1642e-03, +9.1719e-03]
  |U| max:  +7.9288e-02


## Build and launch the Results viewer

`Results.from_fem` wraps the mesh geometry from `fem` together with the numpy fields into a self-contained, session-free object. Calling `results.viewer()` pops the interactive apeGmsh Qt/VTK viewer — use the **Deformation** controls to toggle the deformed shape and the **Contour** dropdown to switch between `|U|`, `Ux`, `Uy`, `Uz`, or the raw `Displacement` vector magnitude.

`blocking=False` runs the viewer in a subprocess so the notebook remains interactive.

In [6]:
# --- LEGACY-API-MIGRATED ---
# Single-snapshot ``point_data`` dict -> (T=1, N) NativeWriter write.
# Vector fields (shape ``(N, 3)``) split into per-axis scalar components.
_legacy_pd = {
        'Displacement': disp,
        '|U|':          u_mag,
        'Ux':           disp[:, 0],
        'Uy':           disp[:, 1],
        'Uz':           disp[:, 2],
    }
_legacy_components = {}
for _legacy_cname, _legacy_cval in _legacy_pd.items():
    _legacy_arr = np.asarray(_legacy_cval)
    if _legacy_arr.ndim == 2 and _legacy_arr.shape[1] in (2, 3):
        for _legacy_i, _legacy_ax in enumerate(["x", "y", "z"][: _legacy_arr.shape[1]]):
            _legacy_components[f"{_legacy_cname}_{_legacy_ax}"] = (
                _legacy_arr[:, _legacy_i].reshape(1, -1)
            )
    else:
        _legacy_components[_legacy_cname] = _legacy_arr.reshape(1, -1)
_legacy_time = np.array([1.0], dtype=float)
_legacy_first = next(iter(_legacy_pd.values()))
_legacy_N = int(np.asarray(_legacy_first).shape[0])
if None:
    _legacy_node_ids = np.asarray(
        fem.nodes.get_ids(pg=(None)[0] if None else None), dtype=np.int64,
    )
    if _legacy_node_ids.size != _legacy_N:
        _legacy_node_ids = np.asarray(fem.nodes.ids, dtype=np.int64)
else:
    _legacy_node_ids = np.asarray(fem.nodes.ids, dtype=np.int64)
if _legacy_node_ids.size != _legacy_N:
    raise RuntimeError(
        f"node_ids has {_legacy_node_ids.size} entries but data "
        f"has {_legacy_N}."
    )
from pathlib import Path as _LegacyPath
from apeGmsh.results.writers import NativeWriter as _LegacyNativeWriter
_legacy_path = _LegacyPath(f"{'column_nts_v4'}_legacy.h5")
if _legacy_path.exists():
    _legacy_path.unlink()
with _LegacyNativeWriter(_legacy_path) as _legacy_nw:
    _legacy_nw.open(fem=fem)
    _legacy_sid = _legacy_nw.begin_stage(name='column_nts_v4', kind="static", time=_legacy_time)
    _legacy_nw.write_nodes(
        _legacy_sid, "partition_0",
        node_ids=_legacy_node_ids, components=_legacy_components,
    )
    _legacy_nw.end_stage()
results = Results.from_native(_legacy_path, fem=fem)

results

<>:26: SyntaxWarning: 'NoneType' object is not subscriptable; perhaps you missed a comma?
<>:26: SyntaxWarning: 'NoneType' object is not subscriptable; perhaps you missed a comma?
C:\Users\nmora\AppData\Local\Temp\ipykernel_49452\2698079634.py:26: SyntaxWarning: 'NoneType' object is not subscriptable; perhaps you missed a comma?
  fem.nodes.get_ids(pg=(None)[0] if None else None), dtype=np.int64,


Results: column_nts_v4_legacy.h5
  FEM: 646 nodes, 3684 elements (snapshot_id=6bb161648ef68e796d35a7c20ced744f)
  Stages (1):
    - stage_0 (column_nts_v4): steps=1, kind=static

In [7]:
results.viewer(blocking=False)

<Popen: returncode: None args: ['C:\\Users\\nmora\\venv\\opensees_venv\\Scri...>

## Cantilever closed-form sanity check

Strong-axis bending of the W section (`Ix = bf·H³/12 − (bf − tw)·h³/12`, `H = 340 mm`):

- `δ_EB = P·L³ / (3·E·Ix)`  (Euler-Bernoulli)
- `δ_T  = δ_EB + P·L / (G·As)`, `As ≈ tw·h`  (Timoshenko, adds shear)

In [8]:
top_id = int(next(iter(fem.nodes.get_ids(target='top'))))
tip_disp = ops.nodeDisp(top_id)

bf, tf, h_web, tw, L = 150.0, 20.0, 300.0, 10.0, 2000.0
H  = 2 * tf + h_web
Ix = (bf * H**3) / 12 - ((bf - tw) * h_web**3) / 12
P  = 1000.0

delta_EB    = P * L**3 / (3 * E * Ix)
G           = E / (2 * (1 + nu))
As          = tw * h_web
delta_shear = P * L / (G * As)
delta_T     = delta_EB + delta_shear

print(f'Top node {top_id} displacement:')
print(f'  ux = {tip_disp[0]:+.4e}   uy = {tip_disp[1]:+.4e}   uz = {tip_disp[2]:+.4e}')
print(f'\nCantilever closed-form:')
print(f'  delta_EB         = {delta_EB:.4e} mm')
print(f'  delta_Timoshenko = {delta_T:.4e} mm')
print(f'  FEM uy           = {tip_disp[1]:+.4e} mm')
print(f'  FEM / EB         = {tip_disp[1] / delta_EB:.4f}')
print(f'  FEM / Timoshenko = {tip_disp[1] / delta_T:.4f}')

Top node 34 displacement:
  ux = -1.6530e-04   uy = +7.8710e-02   uz = +2.4277e-06

Cantilever closed-form:
  delta_EB         = 7.5629e-02 mm
  delta_Timoshenko = 8.4295e-02 mm
  FEM uy           = +7.8710e-02 mm
  FEM / EB         = 1.0407
  FEM / Timoshenko = 0.9337


## 9. Capture results — manual + DomainCapture paths

Two ways to produce a native-HDF5 results file consumable by the
apeGmsh ``ResultsViewer``:

1. **Manual path** — query the live OpenSees domain post-analysis,
   open a ``NativeWriter``, and write nodal displacements yourself.
   Good for one-shot snapshots and post-hoc diagnostics.
2. **DomainCapture path** — declare what to capture with
   ``Recorders().nodes(...)``, hand the spec to a ``DomainCapture``
   context, and call ``cap.step(t=...)`` after each ``ops.analyze``
   (the helper does it for you). Scales to multi-stage, transient,
   modal, and multi-recorder runs.

Both produce a file that ``Results.from_native(path).viewer()`` can
open. The viewer launch is gated on ``APEGMSH_SKIP_VIEWER`` so this
notebook is safe to run under nbconvert / CI.


In [9]:
# --- EOS-WIRING-V1 ---
# Manual path: pull displacements off the live domain, write h5 yourself.
from pathlib import Path
import numpy as np
from apeGmsh.results.writers import NativeWriter

results_manual = Path("example_column_nodeToSurface_v4_manual.h5")
if results_manual.exists():
    results_manual.unlink()

_n = len(fem.nodes.ids)
_ux = np.array([ops.nodeDisp(int(nid), 1) for nid in fem.nodes.ids])
_uy = np.array([ops.nodeDisp(int(nid), 2) for nid in fem.nodes.ids])
_uz = np.array([ops.nodeDisp(int(nid), 3) for nid in fem.nodes.ids])
_components = {
    "displacement_x": _ux.reshape(1, _n),
    "displacement_y": _uy.reshape(1, _n),
    "displacement_z": _uz.reshape(1, _n),
}

with NativeWriter(results_manual) as _nw:
    _nw.open(fem=fem)
    _sid = _nw.begin_stage(name="static", kind="static", time=np.array([1.0]))
    _nw.write_nodes(
        _sid, "partition_0",
        node_ids=np.asarray(fem.nodes.ids, dtype=np.int64),
        components=_components,
    )
    _nw.end_stage()

print(f"manual -> {results_manual} ({results_manual.stat().st_size/1024:.1f} KB)")


manual -> example_column_nodeToSurface_v4_manual.h5 (443.1 KB)


In [10]:
# DomainCapture path: declarative recorders, capture during analyze.
from apeGmsh.solvers.Recorders import Recorders
from apeGmsh.results.capture._domain import DomainCapture

recs = Recorders()
recs.nodes(components="displacement")
recs.nodes(components="reaction_force")
spec = recs.resolve(fem, ndm=3, ndf=6)

results_capture = Path("example_column_nodeToSurface_v4_capture.h5")
if results_capture.exists():
    results_capture.unlink()

with DomainCapture(spec, results_capture, fem, ndm=3, ndf=6) as cap:
    cap.begin_stage("run", kind="static")
    # --- copied from cell 4 ---
    import openseespy.opensees as ops

    ops.wipe()
    ops.model('basic', '-ndm', 3, '-ndf', 6)

    ops.timeSeries('Linear', 1)

    # --- Nodes -----------------------------------------------------------------
    for nid, xyz in fem.nodes.get(target='pg_column'):
        ops.node(nid, *xyz, '-ndf', 3)

    ref = fem.nodes.get(target=['base', 'top'])
    for nid, xyz in ref:
        ops.node(nid, *xyz)

    for nid, xyz in fem.nodes.constraints.phantom_nodes():
        ops.node(nid, *xyz)

    # --- Grounded helper node + zeroLength spring at the base ------------------
    base_id = int(next(iter(fem.nodes.get_ids(target='base'))))
    base_xyz = next(xyz for nid, xyz in ref if nid == base_id)

    GROUND_TAG = 10_000
    ops.node(GROUND_TAG, *base_xyz)
    ops.fix(GROUND_TAG, 1, 1, 1, 1, 1, 1)

    K_SPRING = 1e14
    for i in range(6):
        ops.uniaxialMaterial('Elastic', 100 + i, K_SPRING)

    ZL_TAG = 20_000
    ops.element('zeroLength', ZL_TAG, GROUND_TAG, base_id,
                '-mat', 100, 101, 102, 103, 104, 105,
                '-dir', 1, 2, 3, 4, 5, 6)

    # --- Material + tet elements -----------------------------------------------
    E  = 200e3
    nu = 0.3
    ops.nDMaterial('ElasticIsotropic', 1, E, nu)

    for group in fem.elements.get(element_type='tet4'):
        for eid, conn in group:
            ops.element('FourNodeTetrahedron', int(eid), *[int(n) for n in conn], 1)

    # --- MP constraints --------------------------------------------------------
    for master, slaves in fem.nodes.constraints.rigid_link_groups():
        for slave in slaves:
            ops.rigidLink('beam', int(master), int(slave))

    for pair in fem.nodes.constraints.equal_dofs():
        ops.equalDOF(int(pair.master_node), int(pair.slave_node), *pair.dofs)

    # --- Loads -----------------------------------------------------------------
    ops.pattern('Plain', 1, 1)
    for load in fem.nodes.loads.by_kind('nodal'):
        fx, fy, fz = load.force_xyz  or (0.0, 0.0, 0.0)
        mx, my, mz = load.moment_xyz or (0.0, 0.0, 0.0)
        ops.load(int(load.node_id), fx, fy, fz, mx, my, mz)

    # --- Analysis --------------------------------------------------------------
    ops.constraints('Penalty', 1e15, 1e15)
    ops.numberer('RCM')
    ops.system('UmfPack')
    ops.test('NormDispIncr', 1.0e-6, 10)
    ops.algorithm('Linear')
    ops.integrator('LoadControl', 0.1)
    ops.analysis('Static')

    for i in range(10):
        ok = ops.analyze(1)
        cap.step(t=ops.getTime())
        print(f'Step {i+1}: {"ok" if ok == 0 else f"failed ({ok})"}')
    cap.end_stage()

print(f"capture -> {results_capture} ({results_capture.stat().st_size/1024:.1f} KB)")


Step 1: ok


Step 2: ok
Step 3: ok
Step 4: ok
Step 5: ok
Step 6: ok
Step 7: ok
Step 8: ok
Step 9: ok


Step 10: ok


capture -> example_column_nodeToSurface_v4_capture.h5 (730.7 KB)


In [11]:
# Open the captured run in the apeGmsh ResultsViewer (subprocess,
# non-blocking). Set APEGMSH_SKIP_VIEWER=1 to skip in headless / CI.
import os
from apeGmsh.results import Results
results = Results.from_native(results_capture)
if os.environ.get("APEGMSH_SKIP_VIEWER"):
    print("[skip viewer] APEGMSH_SKIP_VIEWER set")
else:
    handle = results.viewer(blocking=False)
    print(f"viewer pid: {handle.pid}  -- close window to exit.")


[skip viewer] APEGMSH_SKIP_VIEWER set
